In [ ]:
from pathlib import Path
import json
import pickle
import sys
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if (start / "src").exists():
        return start
    if (start.parent / "src").exists():
        return start.parent
    for parent in start.parents:
        if (parent / "src").exists():
            return parent
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_DIR exists: {SRC_DIR.exists()}")

module_name = "autoencoder_anomaly_detector"
module_path = SRC_DIR / f"{module_name}.py"
print(f"Module path: {module_path}")
print(f"Module path exists: {module_path.exists()}")

try:
    from autoencoder_anomaly_detector import (
        classify,
        load_ae_run,
        load_window_run,
        mse_reconstruction_scores,
        plot_thresholds,
        roc_auc_binary,
    )
except ModuleNotFoundError:
    if not module_path.exists():
        raise FileNotFoundError(f"Missing {module_name}.py at {module_path}")
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    autoencoder_anomaly_detector = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(autoencoder_anomaly_detector)
    classify = autoencoder_anomaly_detector.classify
    load_ae_run = autoencoder_anomaly_detector.load_ae_run
    load_window_run = autoencoder_anomaly_detector.load_window_run
    mse_reconstruction_scores = autoencoder_anomaly_detector.mse_reconstruction_scores
    plot_thresholds = autoencoder_anomaly_detector.plot_thresholds
    roc_auc_binary = autoencoder_anomaly_detector.roc_auc_binary

def resolve_path(path_value: str | Path) -> Path:
    path_obj = Path(path_value)
    if path_obj.is_absolute():
        return path_obj
    return (PROJECT_ROOT / path_obj).resolve()

def read_json_if_exists(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())

def list_model_runs(model_root: Path) -> list[dict]:
    runs = []
    if not model_root.exists():
        return runs
    for run_dir in sorted(model_root.iterdir()):
        if not run_dir.is_dir():
            continue
        summary = read_json_if_exists(run_dir / "summary.json")
        evaluation = read_json_if_exists(run_dir / "evaluation.json")
        runs.append({
            "run_name": run_dir.name,
            "run_path": run_dir,
            "summary": summary,
            "evaluation": evaluation,
        })
    return runs

AUTOENCODER_ROOT = PROJECT_ROOT / "models" / "autoencoder"
ISOLATION_ROOT = PROJECT_ROOT / "models" / "isolation_forest"

ae_runs = list_model_runs(AUTOENCODER_ROOT)
iso_runs = list_model_runs(ISOLATION_ROOT)

print(f"Found {len(ae_runs)} autoencoder run(s) and {len(iso_runs)} isolation forest run(s).")

In [ ]:
def build_metrics_df(runs: list[dict], model_type: str) -> pd.DataFrame:
    rows = []
    for run in runs:
        summary = run.get("summary", {})
        evaluation = run.get("evaluation", {})
        test_metrics = evaluation.get("test_metrics", {})
        if not test_metrics:
            continue
        threshold_cfg = summary.get("threshold_config", {})
        method = threshold_cfg.get("method", "unknown")
        percentile = threshold_cfg.get("percentile")
        if method == "val_f1":
            eval_group = "val_f1"
        elif percentile is not None:
            eval_group = f"p{percentile}"
        else:
            eval_group = method
        "source": [
                "def find_run_by_name(runs: list[dict], run_name: str) -> dict:",
                "    for run in runs:",
                "        if run.get(\"run_name\") == run_name:",
                "            return run",
                "    raise ValueError(f\"Run not found: {run_name}\")",
                "",
                "def resolve_plot_path(path_value: str | Path, model_subdir: str) -> Path:",
                "    path_obj = Path(path_value)",
                "    if path_obj.is_absolute():",
                "        return path_obj",
                "    candidate = (PROJECT_ROOT / path_obj).resolve()",
                "    if candidate.exists():",
                "        return candidate",
                "    parts = path_obj.parts",
                "    if len(parts) >= 2 and parts[0] == \"models\":",
                "        candidate = (PROJECT_ROOT / \"models\" / model_subdir / Path(*parts[1:])).resolve()",
                "        if candidate.exists():",
                "            return candidate",
                "    return candidate",
                "",
                "def rel_path(path: Path) -> str:",
                "    try:",
                "        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()",
                "    except Exception:",
                "        return path.as_posix()",
                "",
                "def concat_model_rows(rows: list[dict], out_path: Path):",
                "    fig, axes = plt.subplots(len(rows), 2, figsize=(14, 14), dpi=240)",
                "    if len(rows) == 1:",
                "        axes = np.array([axes])",
                "",
                "    for row_idx, row in enumerate(rows):",
                "        title = row[\"title\"]",
                "        dist_img = plt.imread(row[\"dist_path\"])",
                "        time_img = plt.imread(row[\"time_path\"])",
                "        for col_idx, img in enumerate([dist_img, time_img]):",
                "            ax = axes[row_idx, col_idx]",
                "            ax.imshow(img)",
                "            ax.axis(\"off\")",
                "            if col_idx == 0:",
                "                ax.set_ylabel(title, rotation=0, labelpad=42, va=\"center\", fontsize=11, fontweight=\"semibold\")",
                "",
                "    fig.tight_layout()",
                "    fig.savefig(out_path, bbox_inches=\"tight\")",
                "    print(f\"Saved: {rel_path(out_path)}\")",
                "    plt.show()",
                "",
                "def best_run_for_group(df: pd.DataFrame, eval_group: str) -> pd.Series:",
                "    subset = df[df[\"eval_group\"] == eval_group]",
                "    if subset.empty:",
                "        raise ValueError(f\"No models found for {eval_group}\")",
                "    return subset.sort_values([\"f1\", \"precision\", \"recall\"], ascending=False).iloc[0]",
                "",
                "def load_run_plots(run_name: str, run_lookup: list[dict], model_subdir: str) -> tuple[Path, Path]:",
                "    run_info = find_run_by_name(run_lookup, run_name)",
                "    evaluation = run_info.get(\"evaluation\", {})",
                "    saved_plots = evaluation.get(\"saved_plots\", {})",
                "    if model_subdir == \"autoencoder\":",
                "        dist = resolve_plot_path(saved_plots.get(\"reconstruction_error_distribution\"), model_subdir)",
                "        timeline = resolve_plot_path(saved_plots.get(\"anomaly_timeline_with_fn_density\"), model_subdir)",
                "    else:",
                "        dist = resolve_plot_path(saved_plots.get(\"score_distribution\"), model_subdir)",
                "        timeline = resolve_plot_path(saved_plots.get(\"score_timeline\"), model_subdir)",
                "    return dist, timeline",
                "",
                "best_ae_threshold = best_run_for_group(ae_df, next(g for g in sorted(ae_df[\"eval_group\"].unique()) if g.startswith(\"p\")))",
                "best_ae_valf1 = best_run_for_group(ae_df, \"val_f1\")",
                "best_iso_row = best_run_for_group(iso_df, best_iso[\"eval_group\"])",
                "",
                "ae_threshold_dist, ae_threshold_time = load_run_plots(best_ae_threshold[\"run_name\"], ae_runs, \"autoencoder\")",
                "ae_valf1_dist, ae_valf1_time = load_run_plots(best_ae_valf1[\"run_name\"], ae_runs, \"autoencoder\")",
                "iso_dist, iso_time = load_run_plots(best_iso_row[\"run_name\"], iso_runs, \"isolation_forest\")",
                "",
                "missing_paths = [",
                "    p for p in [ae_threshold_dist, ae_threshold_time, ae_valf1_dist, ae_valf1_time, iso_dist, iso_time] if p is None or not Path(p).exists()",
                "]",
                "if missing_paths:",
                "    raise FileNotFoundError(f\"Missing plot(s): {missing_paths}\")",
                "",
                "comparison_stack_path = COMPARISON_PLOTS_DIR / \"best_models_ordered_rows.png\"",
                "rows = [",
                "    {\"title\": \"Autoencoder (threshold)\", \"dist_path\": ae_threshold_dist, \"time_path\": ae_threshold_time},",
                "    {\"title\": \"Autoencoder (val_f1)\", \"dist_path\": ae_valf1_dist, \"time_path\": ae_valf1_time},",
                "    {\"title\": \"Isolation Forest\", \"dist_path\": iso_dist, \"time_path\": iso_time},",
                "]",
                "concat_model_rows(rows, comparison_stack_path)"
            ]

def add_value_labels(ax, bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.012,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#303030",
        )

def plot_grouped_metric_bars(group: pd.DataFrame, title: str, out_path: Path):
    if group.empty:
        raise ValueError(f"No data for {title}")
    metrics = ["precision", "recall", "f1"]
    colors = ["#3B6FB6", "#E68613", "#2E8B57"]
    labels = group["label"].astype(str).tolist()
    x = np.arange(len(labels))
    width = 0.22
    offsets = [-width, 0.0, width]
    fig, ax = plt.subplots(figsize=(10.5, 4.2), dpi=220)
    bar_sets = []
    for metric, color, offset in zip(metrics, colors, offsets):
        values = group[metric].to_numpy(dtype=float)
        bars = ax.bar(
            x + offset,
            values,
            width=width,
            label=metric.capitalize(),
            color=color,
            alpha=1.0,
            edgecolor="none",
            linewidth=0.0,
            antialiased=False,
        )
        bar_sets.append(bars)
    ax.set_title(title, pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=18, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.03)
    ax.grid(axis="y", alpha=0.20, linestyle="--", linewidth=0.6)
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18), frameon=False)
    for bars in bar_sets:
        add_value_labels(ax, bars)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()

def plot_ae_metric_bars(ae_df: pd.DataFrame):
    p_groups = sorted([g for g in ae_df["eval_group"].unique() if g.startswith("p")])
    if not p_groups:
        raise ValueError("No percentile-based autoencoder runs found.")
    p_group = p_groups[0]
    p_df = ordered_group_df(ae_df, p_group)
    v_df = ordered_group_df(ae_df, "val_f1")

    ae_p_path = COMPARISON_PLOTS_DIR / "ae_metrics_p97_5.png"
    ae_v_path = COMPARISON_PLOTS_DIR / "ae_metrics_val_f1.png"

    plot_grouped_metric_bars(p_df, f"Autoencoder Metrics ({p_group})", ae_p_path)
    plot_grouped_metric_bars(v_df, "Autoencoder Metrics (val_f1)", ae_v_path)
    print(f"Saved: {rel_path(ae_p_path)}")
    print(f"Saved: {rel_path(ae_v_path)}")

def plot_iso_metric_bars_like_ae(iso_df: pd.DataFrame):
    p_groups = sorted([g for g in iso_df["eval_group"].unique() if g.startswith("p")])
    if not p_groups:
        raise ValueError("No percentile-based isolation forest runs found.")
    p_group = p_groups[0]
    p_row = iso_df[iso_df["eval_group"] == p_group].head(1)
    v_row = iso_df[iso_df["eval_group"] == "val_f1"].head(1)
    if p_row.empty or v_row.empty:
        raise ValueError("Need both percentile and val_f1 isolation forest runs.")

    iso_group = pd.DataFrame([
        {"label": p_group, "precision": float(p_row.iloc[0]["precision"]), "recall": float(p_row.iloc[0]["recall"]), "f1": float(p_row.iloc[0]["f1"]),},
        {"label": "val_f1", "precision": float(v_row.iloc[0]["precision"]), "recall": float(v_row.iloc[0]["recall"]), "f1": float(v_row.iloc[0]["f1"]),},
    ])
    iso_path = COMPARISON_PLOTS_DIR / "if_metrics_p97_5_vs_val_f1.png"
    plot_grouped_metric_bars(iso_group, "Isolation Forest Metrics (p97.5 vs val_f1)", iso_path)
    print(f"Saved: {rel_path(iso_path)}")

plot_ae_metric_bars(ae_df)
plot_iso_metric_bars_like_ae(iso_df)

In [ ]:
def select_best_model(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        raise ValueError("No models available for selection.")
    sort_cols = ["f1", "precision", "recall"]
    best = df.sort_values(sort_cols, ascending=False).iloc[0]
    return best

best_ae = select_best_model(ae_df)
best_iso = select_best_model(iso_df)

print("Best autoencoder model (by test F1):")
display(pd.DataFrame([best_ae]))

print("Best isolation forest model (by test F1):")
display(pd.DataFrame([best_iso]))

In [ ]:
def find_run_by_name(runs: list[dict], run_name: str) -> dict:
    for run in runs:
        if run.get("run_name") == run_name:
            return run
    raise ValueError(f"Run not found: {run_name}")

def resolve_plot_path(path_value: str | Path, model_subdir: str) -> Path:
    path_obj = Path(path_value)
    if path_obj.is_absolute():
        return path_obj
    candidate = (PROJECT_ROOT / path_obj).resolve()
    if candidate.exists():
        return candidate
    parts = path_obj.parts
    if len(parts) >= 2 and parts[0] == "models":
        candidate = (PROJECT_ROOT / "models" / model_subdir / Path(*parts[1:])).resolve()
        if candidate.exists():
            return candidate
    return candidate

def rel_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()

def best_run_for_group(df: pd.DataFrame, eval_group: str) -> pd.Series:
    subset = df[df["eval_group"] == eval_group]
    if subset.empty:
        raise ValueError(f"No models found for {eval_group}")
    return subset.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]

def load_run_plots(run_name: str, run_lookup: list[dict], model_subdir: str) -> tuple[Path, Path]:
    run_info = find_run_by_name(run_lookup, run_name)
    evaluation = run_info.get("evaluation", {})
    saved_plots = evaluation.get("saved_plots", {})
    if model_subdir == "autoencoder":
        dist = resolve_plot_path(saved_plots.get("reconstruction_error_distribution"), model_subdir)
        timeline = resolve_plot_path(saved_plots.get("anomaly_timeline_with_fn_density"), model_subdir)
    else:
        dist = resolve_plot_path(saved_plots.get("score_distribution"), model_subdir)
        timeline = resolve_plot_path(saved_plots.get("score_timeline"), model_subdir)
    return dist, timeline

def concat_model_rows(rows: list[dict], out_path: Path):
    fig, axes = plt.subplots(len(rows), 2, figsize=(14, 14), dpi=240)
    if len(rows) == 1:
        axes = np.array([axes])

    for row_idx, row in enumerate(rows):
        title = row["title"]
        dist_img = plt.imread(row["dist_path"])
        time_img = plt.imread(row["time_path"])
        for col_idx, img in enumerate([dist_img, time_img]):
            ax = axes[row_idx, col_idx]
            ax.imshow(img)
            ax.axis("off")
            if col_idx == 0:
                ax.set_ylabel(title, rotation=0, labelpad=42, va="center", fontsize=11, fontweight="semibold")

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    print(f"Saved: {rel_path(out_path)}")
    plt.show()

best_ae_threshold = best_run_for_group(ae_df, next(g for g in sorted(ae_df["eval_group"].unique()) if g.startswith("p")))
best_ae_valf1 = best_run_for_group(ae_df, "val_f1")
best_iso_row = best_run_for_group(iso_df, best_iso["eval_group"])

ae_threshold_dist, ae_threshold_time = load_run_plots(best_ae_threshold["run_name"], ae_runs, "autoencoder")
ae_valf1_dist, ae_valf1_time = load_run_plots(best_ae_valf1["run_name"], ae_runs, "autoencoder")
iso_dist, iso_time = load_run_plots(best_iso_row["run_name"], iso_runs, "isolation_forest")

missing_paths = [
    p for p in [ae_threshold_dist, ae_threshold_time, ae_valf1_dist, ae_valf1_time, iso_dist, iso_time] if p is None or not Path(p).exists()
]
if missing_paths:
    raise FileNotFoundError(f"Missing plot(s): {missing_paths}")

comparison_stack_path = COMPARISON_PLOTS_DIR / "best_models_ordered_rows.png"
comparison_stack_path.parent.mkdir(parents=True, exist_ok=True)
rows = [
    {"title": "Autoencoder (threshold)", "dist_path": ae_threshold_dist, "time_path": ae_threshold_time},
    {"title": "Autoencoder (val_f1)", "dist_path": ae_valf1_dist, "time_path": ae_valf1_time},
    {"title": "Isolation Forest", "dist_path": iso_dist, "time_path": iso_time},
]
concat_model_rows(rows, comparison_stack_path)